In [1]:
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
import os, zipfile
from concurrent.futures import ThreadPoolExecutor

# Authenticate using service account
gauth = GoogleAuth(settings_file="settings.yaml")
gauth.ServiceAuth()
drive = GoogleDrive(gauth)

# Shared folder ID
FOLDER_ID = "1OGSTJEPEYiWBsKTquPRth1H_ul6HKH9G"

# Download directory
os.makedirs("downloads", exist_ok=True)

def download_and_zip_folder(folder):
    folder_name = folder['title']
    folder_id = folder['id']
    folder_path = os.path.join("downloads", folder_name)
    if os.path.exists(folder_path):
        return
    os.makedirs(folder_path, exist_ok=True)

    # List files inside the folder
    sub_files = drive.ListFile({
        'q': f"'{folder_id}' in parents and trashed=false"
    }).GetList()

    for f in sub_files:
        file_path = os.path.join(folder_path, f['title'])
        try:
            f.GetContentFile(file_path)
         
        except Exception as e:
            return f"Error downloading {f['title']}: {e}"
    return
          


# Get folders in the shared folder
folders = drive.ListFile({
    'q': f"'{FOLDER_ID}' in parents and trashed=false and mimeType='application/vnd.google-apps.folder'"
}).GetList()

# Use up to 4 threads (you can increase this if needed)
with ThreadPoolExecutor(max_workers=10) as executor:
    executor.map(download_and_zip_folder, folders)


In [3]:
import torch


print("Number of GPU: ", torch.cuda.device_count())
print("GPU Name: ", torch.cuda.get_device_name())


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

ModuleNotFoundError: No module named 'torch'

THIS CREATES A CSV FILE


In [11]:
import os
import csv
import numpy as np

# Set the root directory
root_dir = 'downloads'  # Adjust if needed

# Set output CSV file
csv_file = 'labels.csv'

# Number of folds (for K-fold cross-validation)
num_folds = 5

# Open CSV file for writing
with open(csv_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['filepath', 'category', 'target', 'fold'])  # Header

    # Create a dictionary to map folder names to numeric indices
    folder_to_index = {folder: idx for idx, folder in enumerate(sorted(os.listdir(root_dir))) if os.path.isdir(os.path.join(root_dir, folder))}
    
    # List to keep track of file paths for fold assignment
    file_paths = []
    
    # Walk through subfolders to collect file paths
    for subdir, dirs, files in os.walk(root_dir):
        for file in files:
            if file.endswith('.wav'):
                label = os.path.basename(subdir)
                folder_index = folder_to_index[label]
                filepath = os.path.join(subdir, file).replace("\\", "/")
                file_paths.append((filepath, label, folder_index))

    # Shuffle file paths to randomize fold assignments
    np.random.shuffle(file_paths)

    # Assign folds and write to CSV
    for idx, (filepath, label, folder_index) in enumerate(file_paths):
        fold_index = idx % num_folds  # Assign fold based on index and num_folds
        writer.writerow([filepath, label, folder_index, fold_index])

print(f"✅ CSV created with folds: {csv_file}")


✅ CSV created with folds: labels.csv


makes csv file without duplicate filename

In [1]:
import os
import csv
import numpy as np
from collections import defaultdict

# Set the root directory
root_dir = 'downloads'
csv_file = 'multiclass_labels.csv'
num_folds = 5

def create_class_name_mapping(root_dir, classes_to_include=None):
    folder_names = sorted([
        folder for folder in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, folder))
    ])
    if classes_to_include:
        folder_names = [f for f in folder_names if f in classes_to_include]
    folder_to_index = {class_name: i for i, class_name in enumerate(folder_names)}
    return folder_to_index, folder_names

# Define included classes
classes_to_include = ['Outside,urbanormanmade','Outside,ruralornatural', 'Waterfall', 'Cricket', 'Television',
                      'Environmentalnoise', 'Trafficnoise,roadwaynoise', 'Vehicle', 'Speech', 'Water', 'Wind',
                       'Rain', 'Thunderstorm', 'Train','Raindrop']

# Create mapping
folder_to_index, all_class_names = create_class_name_mapping(root_dir, classes_to_include)
num_classes = len(all_class_names)

# Dictionary: filename → (filepath, label_vector)
file_label_map = {}

# Track the number of Speech and Animal samples and files that are only Speech or only Animal
speech_count = 0
animal_count = 0

# Collect all valid files
for subdir, dirs, files in os.walk(root_dir):
    label = os.path.basename(subdir)
    if label not in folder_to_index:
        continue  # Skip if label not included

    class_index = folder_to_index[label]
    
    for file in files:
        if not file.endswith('.wav'):
            continue
        filepath = os.path.join(subdir, file).replace("\\", "/")

        if file not in file_label_map:
            label_vector = [0] * num_classes
            file_label_map[file] = (filepath, label_vector)
        
        _, label_vector = file_label_map[file]
        label_vector[class_index] = 1  # Add label

# Now, filter out `Speech`-only and `Animal`-only files if they exceed the limit
filtered_items = []
for filename, (filepath, label_vector) in file_label_map.items():
        filtered_items.append((filename, filepath, label_vector))



# Shuffle the filtered entries
np.random.shuffle(filtered_items)

# Write CSV
with open(csv_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    header = ['filepath', 'fold'] + all_class_names
    writer.writerow(header)

    for idx, (filename, filepath, label_vector) in enumerate(filtered_items):
        fold_index = idx % num_folds
        row = [filepath, fold_index] + label_vector
        writer.writerow(row)

print(f"✅ Multi-label CSV created: {csv_file}")
print(f"📦 Total unique files: {len(filtered_items)}")


✅ Multi-label CSV created: multiclass_labels.csv
📦 Total unique files: 5221


In [2]:
import csv
from collections import defaultdict

# Input CSV file
csv_file = 'multiclass_labels.csv'

# Dictionary to store class counts
class_counts = defaultdict(int)

with open(csv_file, 'r', newline='') as f:
    reader = csv.DictReader(f)
    class_columns = reader.fieldnames[2:]  

    for row in reader:
        for cls in class_columns:
            if row[cls].strip() == '1':
                class_counts[cls] += 1

# Print the counts
print("📊 Class Label Counts:")
for cls in class_columns:
    print(f"{cls}: {class_counts[cls]}")


📊 Class Label Counts:
Cricket: 63
Environmentalnoise: 59
Outside,ruralornatural: 245
Outside,urbanormanmade: 247
Rain: 110
Raindrop: 65
Speech: 4030
Television: 59
Thunderstorm: 60
Trafficnoise,roadwaynoise: 58
Train: 185
Vehicle: 129
Water: 157
Waterfall: 59
Wind: 165


In [37]:
import csv
from collections import defaultdict

input_csv = 'multiclass_labels.csv'
output_csv = 'filtered_multiclass_labels.csv'
max_per_class = 50

# Read CSV
with open(input_csv, newline='', encoding='utf-8') as f:
    reader = list(csv.reader(f))
    header = reader[0]
    rows = reader[1:]

class_names = header[2:]
num_classes = len(class_names)

# Storage
shared_rows = []
single_rows_by_class = defaultdict(list)
class_counts = defaultdict(int)

# Classify rows
for row in rows:
    label_vector = list(map(int, row[2:]))
    label_sum = sum(label_vector)
    
    if label_sum > 1:
        shared_rows.append((row, label_vector))
        for i, val in enumerate(label_vector):
            if val:
                class_counts[i] += 1
    elif label_sum == 1:
        class_index = label_vector.index(1)
        single_rows_by_class[class_index].append((row, label_vector))

# Fill up with single-label rows as needed
final_rows = [r for r, _ in shared_rows]

for class_index in range(num_classes):
    current_count = class_counts[class_index]
    remaining = max_per_class - current_count
    if remaining > 0:
        selected = single_rows_by_class[class_index][:remaining]
        for row, label_vector in selected:
            final_rows.append(row)
            class_counts[class_index] += 1

# Save filtered CSV
with open(output_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(final_rows)

# Summary
print(f"✅ Filtered CSV saved to: {output_csv}")
print(f"📦 Total files: {len(final_rows)}")
for i, class_name in enumerate(class_names):
    print(f"{class_name}: {class_counts[i]}")


✅ Filtered CSV saved to: filtered_multiclass_labels.csv
📦 Total files: 734
Cricket: 50
Environmentalnoise: 50
Outside,ruralornatural: 116
Outside,urbanormanmade: 127
Rain: 92
Raindrop: 54
Speech: 311
Television: 50
Thunderstorm: 54
Trafficnoise,roadwaynoise: 50
Train: 50
Vehicle: 50
Water: 50
Waterfall: 50
Wind: 50


In [34]:
import csv
from collections import defaultdict

# Input CSV file
csv_file = 'filtered_multiclass_labels.csv'

# Dictionary to store class counts
class_counts = defaultdict(int)

with open(csv_file, 'r', newline='') as f:
    reader = csv.DictReader(f)
    class_columns = reader.fieldnames[2:]  

    for row in reader:
        for cls in class_columns:
            if row[cls].strip() == '1':
                class_counts[cls] += 1

# Print the counts
print("📊 Class Label Counts:")
for cls in class_columns:
    print(f"{cls}: {class_counts[cls]}")


📊 Class Label Counts:
Animal: 213
Cricket: 50
Environmentalnoise: 50
Outside,ruralornatural: 129
Outside,urbanormanmade: 131
Rain: 92
Raindrop: 54
Speech: 481
Television: 50
Thunderstorm: 54
Trafficnoise,roadwaynoise: 50
Train: 50
Vehicle: 50
Water: 50
Waterfall: 50
Wind: 50


In [ ]:
import os
import csv
import numpy as np
from collections import defaultdict

# Set the root directory
root_dir = 'downloads'
csv_file = 'multiclass_labels.csv'
num_folds = 5

def create_class_name_mapping(root_dir):
    folder_names = sorted([folder for folder in os.listdir(root_dir)
                           if os.path.isdir(os.path.join(root_dir, folder))])
    folder_to_index = {class_name: i for i, class_name in enumerate(folder_names)}
    return folder_to_index, folder_names

# Create mapping for all classes (no filtering)
folder_to_index, all_class_names = create_class_name_mapping(root_dir)
num_classes = len(all_class_names)

# Dictionary: filename → (filepath, label_vector)
file_label_map = {}

# Collect all valid files
for subdir, dirs, files in os.walk(root_dir):
    label = os.path.basename(subdir)
    if label not in folder_to_index:
        continue  # Skip if label is not recognized

    class_index = folder_to_index[label]
    
    for file in files:
        if not file.endswith('.wav'):
            continue
        filepath = os.path.join(subdir, file).replace("\\", "/")

        if file not in file_label_map:
            label_vector = [0] * num_classes
            file_label_map[file] = (filepath, label_vector)
        
        _, label_vector = file_label_map[file]
        label_vector[class_index] = 1  # Add label

# Add all files to the filtered list
filtered_items = [(filename, filepath, label_vector) for filename, (filepath, label_vector) in file_label_map.items()]

# Shuffle the filtered entries
np.random.shuffle(filtered_items)

# Write CSV
with open(csv_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    header = ['filepath', 'fold'] + all_class_names
    writer.writerow(header)

    for idx, (filename, filepath, label_vector) in enumerate(filtered_items):
        fold_index = idx % num_folds
        row = [filepath, fold_index] + label_vector
        writer.writerow(row)

print(f"✅ Multi-label CSV created: {csv_file}")
print(f"📦 Total unique files: {len(filtered_items)}")


✅ Multi-label CSV created: multiclass_labels.csv
📦 Total unique files: 21211


In [4]:
import os
import librosa
import soundfile
import sys
import threading
import queue

def check_audio_file(filepath, q):
    """
    Checks a single audio file for corruption or incompleteness.
    This function is designed to be run in a separate thread.

    Args:
        filepath (str): The path to the audio file.
        q (queue.Queue): A queue to put the results (errors or None).
    """
    try:
        # Use soundfile first
        try:
            soundfile.read(filepath)
        except Exception as e:
            # If soundfile fails, try librosa
            try:
                librosa.load(filepath, sr=None)
            except Exception as le:
                if "End of file" in str(le):
                    q.put(("incomplete", filepath, le))
                    return
                elif "format is not supported" in str(le):
                    q.put(("unsupported", filepath, le))
                    return
                else:
                    q.put(("corrupted", filepath, le))
                    return
    except Exception as e:
        q.put(("error", filepath, e))
        return
    q.put(None)  # No error, signal success


def check_audio_files(folder_path, num_threads=4):
    """
    Checks audio files in a folder and its subfolders for corruption or incompleteness using threads.

    Args:
        folder_path (str): The path to the folder containing the audio files.
        num_threads (int, optional): The number of threads to use. Defaults to 4.
    """

    if not os.path.exists(folder_path):
        print(f"Error: Folder not found at {folder_path}")
        return [], [] # return empty lists

    supported_formats = ['.wav', '.mp3', '.flac', '.ogg', '.aiff', '.aif']
    corrupted_files = []
    incomplete_files = []
    other_errors = []
    threads = []
    q = queue.Queue()

    for root, _, files in os.walk(folder_path):
        for filename in files:
            if any(filename.lower().endswith(fmt) for fmt in supported_formats):
                filepath = os.path.join(root, filename)
                thread = threading.Thread(target=check_audio_file, args=(filepath, q), name=f"CheckThread-{filename}")
                threads.append(thread)
                thread.start()

    for thread in threads:
        thread.join()

    # Process results from the queue
    while not q.empty():
        result = q.get()
        if result is not None:
            error_type, filepath, error = result
            if error_type == "corrupted":
                corrupted_files.append(filepath)
            elif error_type == "incomplete":
                incomplete_files.append(filepath)
            elif error_type == "unsupported":
                other_errors.append(filepath)
            else:  # "error"
                other_errors.append(filepath)

    return corrupted_files, incomplete_files  # Return lists


if __name__ == "__main__":
    if len(sys.argv) != 2:
        print("Usage: python check_audio.py <folder_path>")
        sys.exit(1)
    folder_path = "C:\\Users\\Nico\\Downloads\\to give\\yamnet transfer learning\\trainingWithGPU\\downloads"
    corrupted, incomplete = check_audio_files(folder_path, num_threads=8)
    if not corrupted and not incomplete:
        print("All files OK")
    else:
        if corrupted:
            print("Corrupted files:")
            for file in corrupted:
                print(file)
        if incomplete:
            print("Incomplete files:")
            for file in incomplete:
                print(file)


ModuleNotFoundError: No module named 'librosa'